# SAMHSA cleaning pipeline

This notebook loads `data/samhsa_filtered_data.csv`, applies the cleaning steps we defined (drop empty columns, standardized location, address flags, trimmed text, stable IDs, phone parsing, missingness report, normalized multi-value fields, validation), and writes:

- `data/samhsa_clean.csv` — cleaned tabular data  
- `data/facilities.json` — array of records for the web app  

**Environment:** From the repo root, use the project venv: `python -m venv .venv` then `source .venv/bin/activate` and `pip install -r requirements.txt`. In Jupyter / VS Code, select the kernel **Python (.venv BHA-website)** or the interpreter `.venv/bin/python`.

**Run Jupyter from the project root** (`BHA-website-2.0`) so paths resolve correctly, or edit `PROJECT_ROOT` in the setup cell.

## 1. Setup: imports and project paths

Import libraries and set paths to the raw CSV and output files. Uses `pandas` for tabular work.

In [ ]:
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

# Assume notebook is run with cwd = project root (BHA-website-2.0)
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "samhsa_filtered_data.csv").exists():
    PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_CSV = DATA_DIR / "samhsa_filtered_data.csv"
CLEAN_CSV = DATA_DIR / "samhsa_clean.csv"
FACILITIES_JSON = DATA_DIR / "facilities.json"

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("RAW_CSV exists:", RAW_CSV.exists())

## 2. Load the raw CSV

Read the file with UTF-8. This preserves SAMHSA comma-separated text fields inside quoted cells.

In [ ]:
df = pd.read_csv(RAW_CSV, dtype=str, keep_default_na=False)
df.columns = [c.strip() for c in df.columns]
print("rows:", len(df), "cols:", len(df.columns))
df.head(2)

## 3. Drop columns that are completely empty

Remove `intake1a` and `intake2a` (100% empty in this extract) so downstream exports stay lean.

In [ ]:
EMPTY_DROP = ["intake1a", "intake2a"]
present = [c for c in EMPTY_DROP if c in df.columns]
df = df.drop(columns=present, errors="ignore")
print("Dropped:", present or "(none — columns missing)")

## 4. Standardize `state`, `zip`, and `city`

- **state**: two-letter uppercase  
- **zip**: digits only, left-padded to 5 characters (string)  
- **city**: strip; **city_display** uses title case for UI labels

In [ ]:
def standardize_zip(z: str) -> str:
    if pd.isna(z) or z == "":
        return ""
    digits = re.sub(r"\D", "", str(z))[:5]
    if len(digits) < 5:
        return digits.zfill(5) if digits else ""
    return digits


def standardize_state(s: str) -> str:
    if pd.isna(s) or s == "":
        return ""
    return str(s).strip().upper()[:2]


df["city_raw"] = df["city"].astype(str).str.strip()
df["city"] = df["city_raw"]
df["city_display"] = df["city"].apply(lambda x: str(x).strip().title() if str(x).strip() else "")
df["state"] = df["state"].apply(standardize_state)
df["zip"] = df["zip"].apply(standardize_zip)

allowed_states = {"DE", "NJ", "NY", "PA"}
bad_state = ~df["state"].isin(allowed_states | {""})
print("rows with unexpected state:", int(bad_state.sum()))
if bad_state.any():
    print(df.loc[bad_state, ["facility_name", "state"]].head(10).to_string())

## 5. Address quality: flags and placeholder handling

Flag empty addresses and SAMHSA placeholder `- - -`. **`address_clean`** is empty when the street was withheld; use it for geocoding logic.

In [ ]:
PLACEHOLDER = "- - -"


def clean_address(a: str) -> str:
    if pd.isna(a):
        return ""
    s = str(a).strip()
    if s == PLACEHOLDER:
        return ""
    return s


df["address"] = df["address"].astype(str).apply(
    lambda x: x.strip() if x.strip() not in ("nan", "None") else ""
)
df["address_clean"] = df["address"].apply(clean_address)
df["address_is_empty"] = df["address_clean"] == ""
df["address_was_placeholder"] = df["address"].astype(str).str.strip() == PLACEHOLDER

print("address_is_empty:", int(df["address_is_empty"].sum()))
print("address_was_placeholder:", int(df["address_was_placeholder"].sum()))

## 6. Trim whitespace on all string columns

Strip leading/trailing spaces on every text field so filters and exports behave consistently.

In [ ]:
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
        df[col] = df[col].replace({"nan": "", "None": ""})

## 7. Add a stable `facility_id`

Computed **after** location and address cleanup so the key reflects normalized `zip`/`state` and trimmed `address`. SHA-256 hex, first 16 characters.

In [ ]:
def make_facility_id(row: pd.Series) -> str:
    parts = [
        str(row.get("facility_name", "")).strip(),
        str(row.get("address", "")).strip(),
        str(row.get("zip", "")).strip(),
        str(row.get("state", "")).strip(),
    ]
    key = "|".join(parts)
    return hashlib.sha256(key.encode("utf-8")).hexdigest()[:16]


df.insert(0, "facility_id", df.apply(make_facility_id, axis=1))
dup_ids = df["facility_id"].duplicated().sum()
print("duplicate facility_id count:", int(dup_ids))
assert dup_ids == 0, "Collision in facility_id — inspect rows"

## 8. Phone: main number and extension

Parse `phone` into **`phone_main`** (formatted when 10 digits) and **`phone_extension`** when patterns like `x1234` appear.

In [ ]:
EXT_RE = re.compile(r"(?i)\s*x\s*([0-9]+)\s*$")


def split_phone(raw: str) -> tuple[str, str, str]:
    """Returns (original_display, phone_main_formatted, extension)."""
    if pd.isna(raw) or str(raw).strip() == "":
        return "", "", ""
    s = str(raw).strip()
    ext = ""
    m = EXT_RE.search(s)
    if m:
        ext = m.group(1)
        s = EXT_RE.sub("", s).strip()
    digits = re.sub(r"\D", "", s)
    if len(digits) == 10:
        main_fmt = f"{digits[:3]}-{digits[3:6]}-{digits[6:]}"
    else:
        main_fmt = s
    return raw.strip(), main_fmt, ext


split = df["phone"].apply(split_phone)
df["phone_display"] = split.apply(lambda t: t[0])
df["phone_main"] = split.apply(lambda t: t[1])
df["phone_extension"] = split.apply(lambda t: t[2])
print("rows with extension:", int((df["phone_extension"] != "").sum()))
df[["phone", "phone_main", "phone_extension"]].head(6)

## 9. Document missingness (per-column report)

Treat empty string as missing. Use this to see sparse domains (do not assume blank means “no service”).

In [ ]:
def is_blank(x) -> bool:
    if pd.isna(x):
        return True
    s = str(x).strip()
    return s == "" or s.lower() in ("none", "nan")


missing_pct = {}
for col in df.columns:
    missing_pct[col] = df[col].apply(is_blank).mean() * 100

miss_df = (
    pd.Series(missing_pct)
    .sort_values(ascending=False)
    .rename("pct_missing")
    .round(2)
    .to_frame()
)
miss_df.head(20)

## 10. Normalize multi-value SAMHSA strings

Split on commas, trim, drop duplicate segments (case-insensitive), sort alphabetically, join with ` | `. Original columns stay unchanged; normalized values are in **`*_norm`** columns.

In [ ]:
MULTI_COLS = [
    "type_of_care",
    "service_setting",
    "treatment_approaches",
    "payment_funding",
    "special_programs_groups",
    "emergency_services",
    "ancillary_services",
    "language_services",
    "age_groups_accepted",
    "pharmacotherapies",
]


def normalize_multivalue(text: str) -> str:
    if pd.isna(text) or str(text).strip() == "":
        return ""
    parts = [p.strip() for p in str(text).split(",")]
    parts = [p for p in parts if p]
    seen: set[str] = set()
    unique: list[str] = []
    for p in parts:
        k = p.lower()
        if k not in seen:
            seen.add(k)
            unique.append(p)
    unique.sort(key=str.lower)
    return " | ".join(unique)


for c in MULTI_COLS:
    if c in df.columns:
        df[c + "_norm"] = df[c].apply(normalize_multivalue)

print("Added *_norm columns:", [c for c in df.columns if c.endswith("_norm")])

## 11. Validation checks

Assert unique `facility_id` and that non-empty ZIPs are five digits. Lists rows if any ZIP fails.

In [ ]:
assert df["facility_id"].is_unique, "facility_id must be unique"

zip_bad = df["zip"].apply(lambda z: z != "" and len(z) != 5)
print("ZIP not length 5 (non-empty):", int(zip_bad.sum()))
if zip_bad.any():
    print(df.loc[zip_bad, ["facility_name", "zip"]].to_string())

assert df.loc[df["zip"] != "", "zip"].str.match(r"^\d{5}$").all(), "ZIP validation failed"

print("OK: validation passed for current extract.")

## 12. Build JSON records for the website

Each row becomes one JSON object with **all** columns from the cleaned dataframe (strings, booleans for flag columns). Empty values are stored as empty strings for simple JS consumption.

In [ ]:
def row_to_json_safe(row: pd.Series) -> dict:
    out = {}
    for col in df.columns:
        v = row[col]
        if pd.isna(v):
            out[col] = ""
        elif isinstance(v, (bool, np.bool_)):
            out[col] = bool(v)
        elif isinstance(v, str):
            out[col] = v
        else:
            out[col] = str(v)
    return out


records = [row_to_json_safe(row) for _, row in df.iterrows()]
print("records:", len(records), "fields per record:", len(records[0]) if records else 0)

## 13. Write `samhsa_clean.csv` and `facilities.json`

Save outputs under `data/`. Re-run after updating the raw CSV.

In [ ]:
df.to_csv(CLEAN_CSV, index=False)
print("Wrote:", CLEAN_CSV)

with open(FACILITIES_JSON, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print("Wrote:", FACILITIES_JSON)